In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
pip install wandb

Mapping 19 labels into 6

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
import numpy as np
import os
import wandb

os.environ["WANDB_API_KEY"] = # Add API key
os.environ["WANDB_PROJECT"] = # Add project name

ADVERSE_LABELS = [
    'adverse_employment', 'adverse_housing', 'adverse_transportation',
    'adverse_parent', 'adverse_relationship', 'adverse_support'
]

ADVERSE_MAPPING = {
    'adverse_employment': ['EMPLOYMENT_unemployed', 'EMPLOYMENT_underemployed', 'EMPLOYMENT_disability'],
    'adverse_housing': ['HOUSING_poor', 'HOUSING_undomiciled', 'HOUSING_other'],
    'adverse_transportation': ['TRANSPORTATION_distance', 'TRANSPORTATION_resource', 'TRANSPORTATION_other'],
    'adverse_parent': ['PARENT'],
    'adverse_relationship': ['RELATIONSHIP_widowed', 'RELATIONSHIP_divorced', 'RELATIONSHIP_single'],
    'adverse_support': ['SUPPORT_minus']
}

SYNTHETIC_LABEL_MAP = {
    'employment': 'adverse_employment', 'housing': 'adverse_housing',
    'transportation': 'adverse_transportation', 'parent': 'adverse_parent',
    'relationship': 'adverse_relationship', 'support': 'adverse_support'
}

Mapping 19 columns into 6

In [19]:
def create_adverse_labels_from_mimic(df):
    df_out = pd.DataFrame(index=df.index)
    df_out['text'] = df['text']

    for adv_label, source_cols in ADVERSE_MAPPING.items():
        numeric_cols = pd.DataFrame()
        for col in source_cols:
            if col in df.columns:
                 numeric_cols[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols = numeric_cols.fillna(0)
        df_out[adv_label] = (numeric_cols.sum(axis=1) > 0).astype(float)

    df_out['labels'] = df_out[ADVERSE_LABELS].values.tolist()
    return df_out[['text', 'labels']].dropna(subset=['text'])

def load_and_format_synthetic_data(filenames):
    df_list = []
    base_path = '/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/'

    for f in filenames:
        file_path = os.path.join(base_path, f)
        try:
            df_list.append(pd.read_csv(file_path))
            print(f"Successfully loaded {file_path}")
        except FileNotFoundError:
            print(f"Warning: Synthetic file {file_path} not found.")
        except Exception:
            if 'SyntheticSentencs_Round2' in f:
                try:
                    typo_path = os.path.join(base_path, 'SyntheticSentencs_Round2.csv')
                    df_list.append(pd.read_csv(typo_path))
                    print(f"Successfully loaded {typo_path} (via typo fix)")
                except: pass

    if not df_list:
        return pd.DataFrame(columns=['text', 'labels'])

    df_synthetic = pd.concat(df_list)
    df_synthetic = df_synthetic.dropna(subset=['text', 'label', 'adverse'])

    df_out = pd.DataFrame(0.0, index=df_synthetic.index, columns=ADVERSE_LABELS)
    df_out['text'] = df_synthetic['text']

    for i, row in df_synthetic.iterrows():
        labels = str(row['label']).split(',')
        adverse_statuses = str(row['adverse']).split(',')

        for j, label_str in enumerate(labels):
            if j >= len(adverse_statuses): continue
            label_str = label_str.strip().lower()
            adverse_str = adverse_statuses[j].strip().lower()

            if adverse_str == 'adverse' and label_str in SYNTHETIC_LABEL_MAP:
                target_label = SYNTHETIC_LABEL_MAP[label_str]
                df_out.at[i, target_label] = 1.0

    df_out['labels'] = df_out[ADVERSE_LABELS].values.tolist()
    return df_out[['text', 'labels']]

Loading and splitting data

In [20]:
try:
    df_mimic = pd.read_csv('/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SDOH_MIMICIII_physio_release.csv')
    df_mimic_adverse = create_adverse_labels_from_mimic(df_mimic)
    print(f"Loaded {len(df_mimic_adverse)} real MIMIC sentences.")
except FileNotFoundError:
    print("Error: Could not find SDOH_MIMICIII_physio_release.csv")

train_val_df, test_df = train_test_split(df_mimic_adverse, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

print(f"Train Size: {len(train_df)} | Val Size: {len(val_df)} | Test Size: {len(test_df)}")

Loaded 5321 real MIMIC sentences.
Train Size: 3192 | Val Size: 1064 | Test Size: 1065


Calculating Class Weights

In [21]:
labels_array = np.array(train_df['labels'].tolist())
pos_counts = np.sum(labels_array, axis=0)
neg_counts = len(train_df) - pos_counts
pos_weight = (neg_counts + 1) / (pos_counts + 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float).to(device)
print(f"Weights calculated for {len(pos_weight)} labels.")

Weights calculated for 6 labels.


Adding synthetic data

In [22]:
synthetic_files = [
    'ManuallyAnnotatedSyntheticSentences.csv',
    'SyntheticSentences_Round1.csv',
    'SyntheticSentencs_Round2.csv'
]
df_synthetic_adverse = load_and_format_synthetic_data(synthetic_files)

train_df_augmented = pd.concat([train_df, df_synthetic_adverse]).sample(frac=1)
print(f"Final Augmented Train Size: {len(train_df_augmented)}")

train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/ManuallyAnnotatedSyntheticSentences.csv
Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SyntheticSentences_Round1.csv
Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SyntheticSentencs_Round2.csv
Final Augmented Train Size: 5472


Model Initialisation with pretrained weights

In [23]:

model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(ADVERSE_LABELS),
    problem_type="multi_label_classification"
)

for param in model.bert.embeddings.parameters():
    param.requires_grad = False
for layer in model.bert.encoder.layer[:10]:
    for param in layer.parameters():
        param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=512)

train_dataset_tokenized = train_dataset.map(tokenize_function, batched=True)
val_dataset_tokenized = val_dataset.map(tokenize_function, batched=True)
test_dataset_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5472 [00:00<?, ? examples/s]

Map:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/1065 [00:00<?, ? examples/s]

Threshold

In [25]:
def compute_metrics_multilabel_optimized(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))

    best_f1 = 0
    best_threshold = 0.5

    for threshold in np.arange(0.1, 0.9, 0.05):
        predictions = (probs > threshold).astype(int)
        f1 = f1_score(y_true=labels, y_pred=predictions, average='micro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    f1_micro = best_f1
    roc_auc_micro = roc_auc_score(y_true=labels, y_score=probs, average='micro')

    return {
        'f1_micro': f1_micro,
        'roc_auc_micro': roc_auc_micro,
        'best_threshold': best_threshold
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

**Training**

In [26]:
training_args = TrainingArguments(
    output_dir='./results_adverse_sdoh_aug',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_dir='./logs_adverse_sdoh_aug',
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=1,
    report_to="wandb"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    compute_metrics=compute_metrics_multilabel_optimized,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,Roc Auc Micro,Best Threshold
1,16.792900,2.035081,0.012278,0.771224,0.850000
2,3.168500,0.658228,0.039264,0.917799,0.800000
3,2.327500,1.029130,0.039574,0.920056,0.350000
4,2.337100,1.044291,0.054054,0.932956,0.350000
5,2.178600,1.196985,0.059361,0.940671,0.300000
6,2.137700,1.579725,0.061321,0.941283,0.150000
7,2.223400,1.548613,0.059770,0.937247,0.200000
8,2.161300,1.580110,0.060345,0.935800,0.150000
9,2.218400,1.478661,0.062500,0.937487,0.200000
10,2.121000,1.551593,0.058700,0.937057,0.150000


TrainOutput(global_step=1710, training_loss=3.7666337398060583, metrics={'train_runtime': 2560.7538, 'train_samples_per_second': 21.369, 'train_steps_per_second': 0.668, 'total_flos': 1.439795402440704e+16, 'train_loss': 3.7666337398060583, 'epoch': 10.0})

Evaluation

In [27]:
test_results = trainer.evaluate(test_dataset_tokenized)
print(f"  Micro F1-Score: {test_results['eval_f1_micro']:.4f}")
print(f"  Micro ROC-AUC:  {test_results['eval_roc_auc_micro']:.4f}")

wandb.finish()

  Micro F1-Score: 0.1264
  Micro ROC-AUC:  0.9396


eval/best_threshold,█▇▃▃▃▁▁▁▁▁█
eval/f1_micro,▁▃▃▄▄▄▄▄▄▄█
eval/loss,█▁▃▃▄▆▆▆▅▆▂
eval/roc_auc_micro,▁▇▇████████
eval/runtime,▃▂▃▂▂▂▃▁▃▄█
eval/samples_per_second,▆▇▆▇▇▇▆█▆▅▁
eval/steps_per_second,▆▇▆▇▇▇▆█▆▅▁
train/epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇████
train/global_step,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇████
train/grad_norm,█▃▃▁▂▆▂▆▁▁
+2,...


In [28]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

print(f"Vocab Size: {config.vocab_size}")
print(f"Hidden Size: {config.hidden_size}")
print(f"Layers: {config.num_hidden_layers}")
print(f"Max Position Embeddings: {config.max_position_embeddings}")

Vocab Size: 28996
Hidden Size: 768
Layers: 12
Max Position Embeddings: 512


In [29]:
from transformers import AutoModel

model_name = "emilyalsentzer/Bio_ClinicalBERT"
model = AutoModel.from_pretrained(model_name)

for param in model.embeddings.parameters():
    param.requires_grad = False
for layer in model.encoder.layer[:10]:
    for param in layer.parameters():
        param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"--- Model Statistics for {model_name} ---")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.1f}%)")
print(f"Frozen Parameters:    {frozen_params:,}")

model_size_mb = total_params * 4 / (1024 * 1024)
print(f"Approximate Model Size: {model_size_mb:.2f} MB")

--- Model Statistics for emilyalsentzer/Bio_ClinicalBERT ---
Total Parameters:     108,310,272
Trainable Parameters: 14,766,336 (13.6%)
Frozen Parameters:    93,543,936
Approximate Model Size: 413.17 MB


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv('/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SDOH_MIMICIII_physio_release.csv')
except FileNotFoundError:
    print("Error: File not found. Using local file name for demonstration.")
    df = pd.read_csv('SDOH_MIMICIII_physio_release.csv')
df_clean = df.dropna(subset=['text'])

train_val_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

total_patients = df_clean['patient_id'].nunique()
train_patients_count = train_df['patient_id'].nunique()
val_patients_count = val_df['patient_id'].nunique()
test_patients_count = test_df['patient_id'].nunique()

print("--- Patient Population Statistics ---")
print(f"Total Unique Patients in Dataset: {total_patients}")
print(f"Unique Patients in Training Set:  {train_patients_count}")
print(f"Unique Patients in Validation Set:{val_patients_count}")
print(f"Unique Patients in Testing Set:   {test_patients_count}")

--- Patient Population Statistics ---
Total Unique Patients in Dataset: 183
Unique Patients in Training Set:  183
Unique Patients in Validation Set:177
Unique Patients in Testing Set:   177
